# 04 · Enrutado de modelos y salida tipada

Hasta ahora el generador era un sustituto. Un sistema RAG real llama a un modelo de lenguaje, y en cuanto lo hace
aparecen dos problemas de ingeniería:

1. **¿Qué modelo, de qué proveedor, y qué pasa cuando falla?** Los proveedores se diferencian en bibliotecas cliente,
   formatos de mensaje, tipos de error y precios, y cualquiera puede agotar el tiempo o limitarte el ritmo.
   **LiteLLM** ofrece una única convención de llamada para todos, además de reintentos y alternativas.
2. **¿Puedes fiarte de la forma de lo que devuelve?** Un modelo devuelve *texto*. Tu código necesita un objeto
   tipado con una respuesta verificada, citas que existan y un indicador de rechazo que sea un booleano.
   **Pydantic** convierte la salida del modelo en un objeto validado o en un error preciso que puedes devolverle.

Todo aquí funciona sin conexión: LiteLLM puede devolver un `mock_response` predefinido a través de la ruta de llamada
real, de modo que el enrutado, el manejo de errores y el código de costes son el código que ejecutarías en producción,
y no hace falta ninguna clave. Donde se llamaría a un modelo alojado, la celda es opcional y está protegida por una
variable de entorno.

**Aprenderás**
1. Una interfaz para muchos modelos con LiteLLM, y cómo una cadena de modelo elige el proveedor
2. Reintentos y alternativas a un segundo modelo cuando falla el primero
3. Contar tokens y estimar el coste por pregunta
4. Modelos Pydantic v2 para peticiones y respuestas, y cómo es un error de validación
5. Salida estructurada del modelo: validar, devolver el error, reintentar y medir con qué frecuencia eso rescata una mala respuesta
6. Un esquema de respuesta RAG cuyas citas se comprueban contra los pasajes que realmente se recuperaron

## 1. Una interfaz, muchos modelos

Una cadena de modelo de LiteLLM es `proveedor/modelo`. La llamada y la respuesta tienen la forma de OpenAI sea cual sea
el proveedor. En un agente de analítica de parques eólicos de este portafolio, cada llamada de texto pasaba por LiteLLM
con un único ajuste (`LLM_PROVIDER`) que elegía el modelo: Amazon Nova en AWS Bedrock, con el perfil de inferencia de
la UE para que las peticiones se queden en Europa. Cambiar de modelo era un cambio de configuración, no de código.

| Proveedor | Cadena de modelo (ejemplos) | Requiere |
|---|---|---|
| AWS Bedrock | `bedrock/eu.amazon.nova-lite-v1:0` | credenciales de AWS, una región |
| OpenAI | `openai/gpt-4o-mini` | `OPENAI_API_KEY` |
| Anthropic | `anthropic/claude-...` | `ANTHROPIC_API_KEY` |
| Un servidor local | `ollama/llama3` | un servidor Ollama en tu máquina |

Aquí está la llamada idéntica para tres proveedores distintos. Solo cambia la cadena del modelo, y `mock_response`
sustituye a la red para que funcione sin conexión.

In [1]:
import os, re, json, time, math, warnings
import numpy as np
warnings.filterwarnings("ignore")
import litellm
litellm.suppress_debug_info = True

messages = [{"role": "user", "content": "Say hello in five words."}]
for model in ["bedrock/eu.amazon.nova-lite-v1:0", "openai/gpt-4o-mini", "ollama/llama3"]:
    r = litellm.completion(model=model, messages=messages, mock_response="Hello, how can I help?")
    print(f"{model:36s} -> {r.choices[0].message.content!r}   finish_reason={r.choices[0].finish_reason}")
print("\nfields every response has:", [k for k in ("id", "model", "choices", "usage") if hasattr(r, k)])

bedrock/eu.amazon.nova-lite-v1:0     -> 'Hello, how can I help?'   finish_reason=stop
openai/gpt-4o-mini                   -> 'Hello, how can I help?'   finish_reason=stop
ollama/llama3                        -> 'Hello, how can I help?'   finish_reason=stop

fields every response has: ['id', 'model', 'choices', 'usage']


## 2. Cuando el modelo falla: reintentos y alternativas

Los proveedores reales limitan el ritmo (HTTP 429), agotan el tiempo y sufren caídas. LiteLLM lanza las *mismas*
clases de excepción para todos (`RateLimitError`, `Timeout`, `ServiceUnavailableError`...), así que tu manejo se
escribe una sola vez. Su `Router` añade la política: reintentar el mismo despliegue unas cuantas veces y después pasar
a otro. Simulamos un modelo principal que siempre limita el ritmo y uno de reserva que responde.

In [2]:
from litellm import Router

try:
    litellm.completion(model="openai/gpt-4o-mini", messages=messages, mock_response="litellm.RateLimitError")
except Exception as e:
    print("one call, no policy ->", type(e).__name__)

router = Router(
    model_list=[
        {"model_name": "primary", "litellm_params": {"model": "openai/gpt-4o-mini", "mock_response": "litellm.RateLimitError"}},
        {"model_name": "backup",  "litellm_params": {"model": "bedrock/eu.amazon.nova-lite-v1:0", "mock_response": "Answer from the backup model."}},
    ],
    fallbacks=[{"primary": ["backup"]}], num_retries=1,
)
t0 = time.perf_counter()
r = router.completion(model="primary", messages=messages)
print(f"router with fallback -> {r.choices[0].message.content!r} served by {r.model} in {time.perf_counter() - t0:.2f} s")

one call, no policy -> RateLimitError


router with fallback -> 'Answer from the backup model.' served by eu.amazon.nova-lite-v1:0 in 2.34 s


Dos notas de diseño. Primero, una alternativa debería ser normalmente un modelo de *capacidad similar*: una alternativa
mucho más débil convierte una caída en respuestas peores sin avisar, así que registra qué despliegue atendió cada
petición (el campo `model` de arriba) y ponlo en tus trazas (módulo 09). Segundo, reintenta solo los errores que pueden
tener éxito en un segundo intento: un límite de ritmo o un tiempo agotado sí, una petición mal formada no. Las clases de
excepción de LiteLLM permiten esa distinción sin código específico de cada proveedor.

## 3. Tokens y coste por pregunta

LiteLLM mantiene una tabla de precios por modelo y cuenta tokens, así que puedes estimar lo que cuesta una pregunta RAG
*antes* de llamar a nada. El prompt son los pasajes recuperados más la pregunta; la respuesta es corta. (Los precios se
leen de la tabla de LiteLLM al ejecutar y cambian; consulta la página de tu proveedor antes de presupuestar. La llamada
simulada devuelve recuentos de tokens de relleno, así que contamos los tokens nosotros.)

In [3]:
# El manual de Harbourview: 22 pasajes, ficticios, escritos para este curso
PASSAGES = [
    ["p01", "Safety / Access", "Technicians must hold a valid GWO Basic Safety Training certificate before boarding a crew transfer vessel. Certificates are checked at the harbour gate and expire after two years."],
    ["p02", "Safety / Weather limits", "Crew transfer vessels do not sail when the significant wave height exceeds 1.5 m or the wind speed at the harbour exceeds 20 m/s. The duty coordinator decides at 06:00 and may cancel later."],
    ["p03", "Safety / Lone working", "Lone working inside a nacelle is forbidden. Every task inside a turbine needs at least two people, and both must carry a personal locator beacon."],
    ["p04", "Turbines / Specification", "Each of the 40 turbines is rated at 8.0 MW with a rotor diameter of 164 m. Cut-in wind speed is 3 m/s, rated speed is 12.5 m/s and cut-out speed is 25 m/s."],
    ["p05", "Turbines / Naming", "Turbines are named HV01 to HV40, numbered from the western edge of the array. The offshore substation is OSS1, in the centre."],
    ["p06", "Maintenance / Schedule", "Every turbine gets a planned service once a year between April and September. The service takes two technicians about two days, plus weather delays."],
    ["p07", "Maintenance / Gearbox oil", "Gearbox oil is sampled every six months. If the particle count exceeds ISO 4406 code 18/16/13, a filter change is scheduled within 14 days."],
    ["p08", "Maintenance / Blade inspection", "Blades are inspected by drone every 12 months and after any lightning alarm. Leading-edge erosion above grade 3 is repaired in the next planned service."],
    ["p09", "Faults / Alarm levels", "Alarms have three levels. Level 1 is informational. Level 2 needs a remote check within 4 hours. Level 3 stops the turbine and needs a site visit within 24 hours when the weather allows."],
    ["p10", "Faults / Pitch system", "A pitch system fault (code PF-203) usually means a low accumulator pressure. Reset it remotely once. If it returns within 48 hours, send a crew to replace the hydraulic accumulator."],
    ["p11", "Faults / Yaw misalignment", "A yaw error above 8 degrees for more than 10 minutes raises alarm YW-110. Persistent errors point to a faulty wind vane: the vane is recalibrated, then replaced if the error remains."],
    ["p12", "Faults / Grid loss", "On loss of grid connection the turbines stop and the converters isolate. Restart is allowed only when the grid operator confirms voltage within 5% of nominal for 10 minutes."],
    ["p13", "Environment / Marine mammals", "Piling and heavy cable work pause when a marine mammal is sighted within 500 m. Work resumes after 30 minutes without a sighting."],
    ["p14", "Environment / Bird collision", "Between October and March, turbines HV01 to HV08 are curtailed at night during the peak migration periods announced by the ornithology team."],
    ["p15", "Environment / Waste", "Used oil, filters and composite offcuts are brought ashore and logged in the waste register. Nothing is discharged at sea."],
    ["p16", "Logistics / Spare parts", "The harbour warehouse keeps the critical spares: two pitch accumulators, four wind vanes and a spare converter module. Reorder when stock falls below one unit."],
    ["p17", "Logistics / Helicopter", "A helicopter is used only for medical evacuation and for urgent Level 3 visits when vessels cannot sail. Booking goes through the duty coordinator."],
    ["p18", "Contacts / Duty coordinator", "The duty coordinator is reachable 24 hours a day on the control room line. Out-of-hours safety incidents are reported there first."],
    ["p19", "Reporting / Incidents", "Any injury or near miss is reported within one hour and a written report is filed within 24 hours. Environmental incidents follow the same deadlines."],
    ["p20", "Reporting / Performance", "Availability is reported monthly. The target is 97% time-based availability, counting weather downtime as available."],
    ["p21", "Grid / Export", "Power is exported at 66 kV through two cables to the onshore substation. The export limit agreed with the grid operator is 300 MW."],
    ["p22", "Training / Induction", "New staff complete a two-day induction covering the handbook, emergency drills and the sea survival course before their first transfer."],
]

In [4]:
ids = [p[0] for p in PASSAGES]
passage = {pid: (section, text) for pid, section, text in PASSAGES}

def build_prompt(question, retrieved_ids):
    context = "\n".join(f"[{pid}] {passage[pid][0]}: {passage[pid][1]}" for pid in retrieved_ids)
    return [{"role": "system", "content": "Answer ONLY from the passages. Cite passage ids in brackets. If the answer is not there, refuse."},
            {"role": "user", "content": f"PASSAGES:\n{context}\n\nQUESTION: {question}"}]

n_out = 60
for k in (3, 10):
    prompt = build_prompt("What wave height stops the crew boats from sailing?", ids[:k])
    n_in = litellm.token_counter(model="openai/gpt-4o-mini", messages=prompt)
    print(f"{k} retrieved passages: prompt {n_in} tokens, expected answer ~{n_out} tokens")
    for model in ["bedrock/eu.amazon.nova-lite-v1:0", "openai/gpt-4o-mini", "ollama/llama3"]:
        cin, cout = litellm.cost_per_token(model=model, prompt_tokens=n_in, completion_tokens=n_out)
        share = f"{cin / (cin + cout):4.0%} of it is the prompt" if cin + cout else "free per token"
        print(f"  {model:36s} ${cin + cout:.6f} per question = ${(cin + cout) * 1000:5.2f} per 1,000 ({share})")

3 retrieved passages: prompt 175 tokens, expected answer ~60 tokens
  bedrock/eu.amazon.nova-lite-v1:0     $0.000032 per question = $ 0.03 per 1,000 ( 42% of it is the prompt)
  openai/gpt-4o-mini                   $0.000062 per question = $ 0.06 per 1,000 ( 42% of it is the prompt)
  ollama/llama3                        $0.000000 per question = $ 0.00 per 1,000 (free per token)
10 retrieved passages: prompt 497 tokens, expected answer ~60 tokens
  bedrock/eu.amazon.nova-lite-v1:0     $0.000057 per question = $ 0.06 per 1,000 ( 67% of it is the prompt)
  openai/gpt-4o-mini                   $0.000111 per question = $ 0.11 per 1,000 ( 67% of it is the prompt)
  ollama/llama3                        $0.000000 per question = $ 0.00 per 1,000 (free per token)


Lee el desglose, no solo el total. Los tokens de salida cuestan varias veces más por token que los de entrada, así que
con 3 pasajes cortos el coste se reparte entre el prompt y la respuesta; cuando la recuperación envía más texto, el
**prompt pasa a dominar** (compara los dos bloques). Por eso la calidad de la recuperación también es una palanca de
coste: enviar 3 pasajes buenos en lugar de 10 mediocres reduce la factura y suele mejorar la respuesta. Un modelo local
no cuesta nada por token (pagas en hardware y latencia). Sea cual sea el modelo, una pregunta RAG sobre un corpus de
pasajes cortos como este cuesta una pequeña fracción de céntimo.

## 4. Pydantic: peticiones y respuestas tipadas

**Pydantic** v2 valida datos frente a las anotaciones de tipo de Python y sus restricciones, y te dice con precisión qué
está mal. En un servicio RAG aparece en tres fronteras: la petición de la API, la salida estructurada del modelo y la
respuesta de la API. Primero, el lado de la petición.

In [5]:
from typing import Annotated, Literal
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

class AskRequest(BaseModel):
    question: Annotated[str, Field(min_length=3, max_length=500)]
    k: int = Field(default=3, ge=1, le=10)                  # pasajes a recuperar
    language: Literal["en", "es"] = "en"

print(AskRequest(question="What is the export limit?"))
print(AskRequest.model_validate({"question": "wave limit?", "k": "5"}))      # "5" se convierte a 5
for bad in [{"question": "hi"}, {"question": "ok question", "k": 50}, {"question": "ok question", "language": "fr"}]:
    try:
        AskRequest.model_validate(bad)
    except ValidationError as e:
        err = e.errors()[0]
        print(f"rejected {bad}: field {err['loc'][0]!r}: {err['msg']}")

question='What is the export limit?' k=3 language='en'
question='wave limit?' k=5 language='en'
rejected {'question': 'hi'}: field 'question': String should have at least 3 characters
rejected {'question': 'ok question', 'k': 50}: field 'k': Input should be less than or equal to 10
rejected {'question': 'ok question', 'language': 'fr'}: field 'language': Input should be 'en' or 'es'


Cada error nombra el campo y la regla, en una estructura que un programa puede leer (`e.errors()`). Esa propiedad es la
que hace posible la sección siguiente: el error no es solo un fallo, es *retroalimentación* que puedes devolver al
modelo.

## 5. Salida estructurada: validar, devolver el error, reintentar

Se pide a los modelos que respondan en JSON y casi siempre obedecen. "Casi" es el problema: un campo que falta, un
número como cadena, JSON envuelto en prosa o en un bloque de Markdown. El patrón robusto es un bucle: preguntar, validar
con el modelo Pydantic y, si falla, enviar al modelo **su propia salida más el error de validación** y preguntar de
nuevo, hasta un pequeño límite. El esquema se genera a partir de la propia clase, de modo que el prompt y el validador
nunca pueden discrepar.

In [6]:
class Answer(BaseModel):
    answer: str = Field(min_length=1)
    citations: list[str]                       # ids de pasaje como "p02"
    refused: bool = False

print(json.dumps(Answer.model_json_schema()["properties"], indent=1)[:420], "...")

def extract_json(text):
    # los modelos suelen envolver el JSON en un bloque de Markdown o añadir una frase alrededor: se toma el {...} más externo
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1] if start != -1 and end > start else text

def structured_call(model_cls, messages, outputs, model="bedrock/eu.amazon.nova-lite-v1:0", max_attempts=3, context=None):
    # `outputs` es la lista de respuestas predefinidas que da el modelo simulado en los intentos 1, 2, ... (un modelo real simplemente se llamaría)
    messages = list(messages)
    for attempt in range(1, max_attempts + 1):
        reply = litellm.completion(model=model, messages=messages, mock_response=outputs[min(attempt - 1, len(outputs) - 1)])
        raw = reply.choices[0].message.content
        try:
            return model_cls.model_validate_json(extract_json(raw), context=context), attempt
        except ValidationError as e:
            problems = "; ".join(f"{'.'.join(map(str, x['loc'])) or 'output'}: {x['msg']}" for x in e.errors())
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your reply was invalid ({problems}). Reply again with ONLY valid JSON."}]
    return None, max_attempts

good = '{"answer": "Vessels do not sail above 1.5 m significant wave height.", "citations": ["p02"]}'
bad_json = '{"answer": "Vessels do not sail above 1.5 m", "citations": ["p02"],}'      # coma final
missing = '{"answer": "Vessels do not sail above 1.5 m."}'                                # sin el campo citations
fenced = "Sure! Here is the JSON:\n```json\n" + good + "\n```"

for label, outputs in [("valid first time", [good]), ("trailing comma, then valid", [bad_json, good]),
                       ("missing field, then valid", [missing, good]), ("wrapped in a fence", [fenced]),
                       ("never valid", [bad_json])]:
    obj, attempts = structured_call(Answer, messages, outputs)
    print(f"{label:28s} -> {'ok ' + str(obj.citations) if obj else 'FAILED':14s} after {attempts} attempt(s)")

{
 "answer": {
  "minLength": 1,
  "title": "Answer",
  "type": "string"
 },
 "citations": {
  "items": {
   "type": "string"
  },
  "title": "Citations",
  "type": "array"
 },
 "refused": {
  "default": false,
  "title": "Refused",
  "type": "boolean"
 }
} ...
valid first time             -> ok ['p02']     after 1 attempt(s)
trailing comma, then valid   -> ok ['p02']     after 2 attempt(s)
missing field, then valid    -> ok ['p02']     after 2 attempt(s)
wrapped in a fence           -> ok ['p02']     after 1 attempt(s)
never valid                  -> FAILED         after 3 attempt(s)


El bucle convierte la mayoría de los deslices de formato en una llamada extra, y una respuesta que *nunca* es válida
falla de forma visible tras `max_attempts` en lugar de corromper el código posterior: quien llama recibe `None` y puede
recurrir a un rechazo o a un modelo más seguro. ¿Cuánto compra realmente el reintento? Simula un modelo con deslices
realistas y cuenta.

In [7]:
rng = np.random.default_rng(7)
SLIPS = [bad_json, missing, fenced.replace(good, good.replace('"citations": ["p02"]', '"citations": "p02"')), "I think the limit is 1.5 m."]
trials = 400
first_ok = retried_ok = failed = 0
for _ in range(trials):
    # un modelo cuya respuesta es un desliz el 20 % de las veces en cada intento, de forma independiente
    outputs = [rng.choice(SLIPS) if rng.random() < 0.20 else good for _ in range(3)]
    obj, attempts = structured_call(Answer, messages, outputs)
    if obj is None: failed += 1
    elif attempts == 1: first_ok += 1
    else: retried_ok += 1
print(f"{trials} simulated requests, 20% slip rate per attempt:")
print(f"  valid on the first attempt : {first_ok / trials:6.1%}")
print(f"  rescued by validate + retry: {retried_ok / trials:6.1%}")
print(f"  still invalid after 3      : {failed / trials:6.1%}   (theory: 0.2^3 = {0.2 ** 3:.1%})")

400 simulated requests, 20% slip rate per attempt:
  valid on the first attempt :  75.8%
  rescued by validate + retry:  22.5%
  still invalid after 3      :   1.8%   (theory: 0.2^3 = 0.8%)


El reintento es un seguro barato: la proporción que sigue mal tras tres intentos baja de la tasa de deslices a algo
cercano a su cubo (aquí algo por encima del valor teórico, lo que es ruido de muestreo normal con 400 pruebas). Fíjate
en lo que esto **no** arregla: una respuesta puede estar perfectamente formada y aun así ser incorrecta. Validar la forma
no dice nada sobre la verdad. La sección siguiente añade la comprobación barata más potente que existe para RAG.

## 6. Una respuesta RAG con las citas verificadas

La alucinación RAG más común que un esquema *sí* puede detectar es la cita inventada: el modelo escribe `[p17]` para un
pasaje que nunca vio, o no cita nada mientras afirma responder. Podemos rechazar ambas pasando los **ids recuperados como
contexto de validación**: un validador de Pydantic los lee y rechaza cualquier cita que no esté entre ellos.

In [8]:
class GroundedAnswer(BaseModel):
    answer: str = Field(min_length=1)
    citations: list[str]
    refused: bool = False

    @field_validator("citations")
    @classmethod
    def cited_passages_were_retrieved(cls, value, info):
        retrieved = (info.context or {}).get("retrieved", [])
        unknown = [c for c in value if c not in retrieved]
        if unknown:
            raise ValueError(f"cites passages that were not retrieved: {unknown} (retrieved: {retrieved})")
        return value

    @model_validator(mode="after")
    def answers_need_evidence(self):
        if not self.refused and not self.citations:
            raise ValueError("an answer must cite at least one passage, or set refused=true")
        return self

retrieved = ["p02", "p09", "p01"]
cases = {
    "cites a retrieved passage": '{"answer": "No sailing above 1.5 m.", "citations": ["p02"]}',
    "invented citation":         '{"answer": "No sailing above 1.5 m.", "citations": ["p17"]}',
    "answer without evidence":   '{"answer": "No sailing above 1.5 m.", "citations": []}',
    "honest refusal":            '{"answer": "I cannot find that in the handbook.", "citations": [], "refused": true}',
}
for label, raw in cases.items():
    try:
        GroundedAnswer.model_validate_json(raw, context={"retrieved": retrieved})
        print(f"accepted  {label}")
    except ValidationError as e:
        print(f"REJECTED  {label}: {e.errors()[0]['msg'].removeprefix('Value error, ')}")

accepted  cites a retrieved passage
REJECTED  invented citation: cites passages that were not retrieved: ['p17'] (retrieved: ['p02', 'p09', 'p01'])
REJECTED  answer without evidence: an answer must cite at least one passage, or set refused=true
accepted  honest refusal


In [9]:
# el mismo bucle de reintento, ahora con la comprobación de fundamento: el modelo primero inventa una cita y luego la corrige
invented = '{"answer": "No sailing above 1.5 m.", "citations": ["p17"]}'
corrected = '{"answer": "No sailing above 1.5 m.", "citations": ["p02"]}'
prompt = build_prompt("What wave height stops the crew boats from sailing?", retrieved)
obj, attempts = structured_call(GroundedAnswer, prompt, [invented, corrected], context={"retrieved": retrieved})
print(f"accepted after {attempts} attempt(s): {obj}")

accepted after 2 attempt(s): answer='No sailing above 1.5 m.' citations=['p02'] refused=False


Un esquema fundamentado detecta *referencias fabricadas y falta de evidencia*; no puede decir si el pasaje `p02` respalda
de verdad la frase. Esa es otra comprobación (la fidelidad), que hace un modelo juez en el módulo 07 y, de forma
continua, el supervisor del módulo 08. Combínalas en capas: los tipos y el fundamento son gratis e instantáneos, juzgar
cuesta una llamada al modelo.

### Opcional: la misma llamada contra un modelo real

Si tienes credenciales, puedes apuntar el código sin cambios a un modelo alojado. La celda de abajo solo se ejecuta si
defines `RAG_LIVE_MODEL` (por ejemplo `bedrock/eu.amazon.nova-lite-v1:0`); el notebook ejecutado no la define, así que
imprime un aviso y no hace ninguna llamada de red.

In [10]:
live_model = os.environ.get("RAG_LIVE_MODEL")
if live_model:
    r = litellm.completion(model=live_model, messages=build_prompt("What wave height stops the crew boats from sailing?", retrieved)
                           + [{"role": "user", "content": 'Reply ONLY with JSON: {"answer": str, "citations": [ids], "refused": bool}'}])
    print(GroundedAnswer.model_validate_json(extract_json(r.choices[0].message.content), context={"retrieved": retrieved}))
else:
    print("RAG_LIVE_MODEL is not set: skipping the real call (everything above ran offline).")

RAG_LIVE_MODEL is not set: skipping the real call (everything above ran offline).


### Una nota de la práctica: cuando LiteLLM no es la capa adecuada

LiteLLM normaliza bien el chat de *texto*. En el agente de parques eólicos, la entrada de imágenes a Amazon Nova a través
de LiteLLM era inconsistente, así que la llamada de visión usó directamente la API Converse de Bedrock mientras todas las
llamadas de texto seguían en LiteLLM. Una capa de enrutado es una comodidad para el camino común, no un muro: conserva una
salida de emergencia al cliente nativo para la única función que la abstracción no cubre, y mantén esa excepción en una
función pequeña.

## Ejercicios

1. Añade `confidence: float = Field(ge=0, le=1)` a `GroundedAnswer`. ¿Cuáles de las respuestas predefinidas de la
   sección 6 fallan ahora, y qué mensaje de error envía el bucle de reintento al modelo?
2. Sube la tasa de deslices simulada de la sección 5 al 50 % y `max_attempts` a 5. ¿La tasa de fallos medida sigue
   `0.5 ** 5`? ¿Qué dice eso sobre confiar en los reintentos con un modelo que casi siempre se equivoca?
3. Añade un tercer despliegue al `Router` y una segunda alternativa. Haz que la primera reserva también falle
   (`mock_response="litellm.Timeout"`) y confirma que la petición la atiende la segunda.
4. Calcula el coste mensual de 20.000 preguntas al mes con 3 y con 8 pasajes recuperados para cada modelo de la
   sección 3. ¿A partir de qué punto un modelo más barato supera a un mejor recuperador en coste, y en calidad?